# 02 - Neural decoder vs MWPM

**In plain words.** Instead of a hand-written rule, we *train* a small neural network on millions of simulated syndromes (the right answer is free in simulation). Question: *when does learning beat the classic decoder?* Our hypothesis: tie on simple independent noise, win when errors come in correlated pairs that MWPM assumes away.

In [1]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
QUICK = True   # False = bigger budgets (slower, tighter error bars)
import numpy as np, tempfile
from src.experiments import Config, get_nn, mwpm_for, fresh_test_data, best_mwpm
from src.noise import NoiseSpec
from src.decoders import GreedyDecoder
from src.evaluate import evaluate_decoders
cfg = Config.quick(tempfile.mkdtemp(), verbose=True) if QUICK else Config(results_dir=pathlib.Path(tempfile.mkdtemp()), verbose=True)
if QUICK: cfg.steps = {3: 800, 5: 800, 7: 800}; cfg.shots_test = 100_000
d, p = 3, 0.02

## Uniform noise (control) - expect a tie

In [2]:
spec = NoiseSpec.uniform(p)
decs = {"mwpm_matched": mwpm_for("rep", d, d, spec), "nn": get_nn(cfg, "rep", d, d, spec), "greedy": GreedyDecoder(d, d)}
det, obs = fresh_test_data(cfg, "rep", d, d, spec)
import pandas as pd
pd.DataFrame(evaluate_decoders(decs, det, obs, rounds=d, reference="mwpm_matched"))[["decoder","ler","ler_lo","ler_hi","adv","adv_lo","adv_hi"]]

[train] rep_uniform_p0.02_d3_r3_mlp.pt: 800 steps, batch 4096, n_det=8


        done in 19s  best val BCE 0.07367 @ step 800


,decoder,ler,ler_lo,ler_hi,adv,adv_lo,adv_hi
0,mwpm_matched,0.02639,0.025415,0.027402,0.00000,0.000000,0.000000
1,nn,0.02478,0.023835,0.025762,0.00161,0.001153,0.002067
2,greedy,0.03198,0.030907,0.033089,-0.00559,-0.006466,-0.004714


## Correlated noise - the interesting case
`adv = LER(best MWPM) - LER(NN)`; positive and with a confidence interval above 0 means the network is genuinely better.

In [3]:
spec_b = NoiseSpec.biased(p, bias=5.0, corr=0.5)
decs = {"mwpm_matched": mwpm_for("rep", d, d, spec_b), "mwpm_uniform": mwpm_for("rep", d, d, spec),
        "nn": get_nn(cfg, "rep", d, d, spec_b), "nn_transfer": get_nn(cfg, "rep", d, d, spec)}
det, obs = fresh_test_data(cfg, "rep", d, d, spec_b)
ref = best_mwpm(cfg, "rep", d, d, spec_b, {k: decs[k] for k in ("mwpm_matched", "mwpm_uniform")})
print("reference MWPM weighting (chosen on validation data):", ref)
pd.DataFrame(evaluate_decoders(decs, det, obs, rounds=d, reference=ref))[["decoder","ler","ler_lo","ler_hi","adv","adv_lo","adv_hi"]]

[train] rep_biased_p0.02_r5_c0.5_d3_r3_mlp.pt: 800 steps, batch 4096, n_det=8


        done in 16s  best val BCE 0.22039 @ step 500


reference MWPM weighting (chosen on validation data): mwpm_uniform


,decoder,ler,ler_lo,ler_hi,adv,adv_lo,adv_hi
0,mwpm_matched,0.13441,0.132310,0.136538,-0.03749,-0.039421,-0.035559
1,mwpm_uniform,0.09692,0.095102,0.098769,0.00000,0.000000,0.000000
2,nn,0.09519,0.093387,0.097025,0.00173,0.001213,0.002247
3,nn_transfer,0.09535,0.093545,0.097186,0.00157,0.001058,0.002082
